# 03 · Feature Policy and Preparation

Define what the model may use, what remains available only for audit and how temporal leakage is prevented.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
base = pd.read_csv(DATA_ROOT / 'alert_data' / 'Base.csv').dropna().copy()

## 1. Feature roles

In [ ]:
target = "fraud_bool"
time_column = "month"
audit_only = ["income", "customer_age", "employment_status", "housing_status"]
categorical = ["payment_type", "source", "device_os"]
pd.Series({
    "target": target,
    "time_column": time_column,
    "audit_only": audit_only,
    "model_categorical": categorical,
})

Income, age, employment and housing are excluded from the primary model and retained for subgroup auditing. This does not remove all possible proxies and must not be described as proof of fairness.

## 2. Temporal split

In [ ]:
train = base[base.month.isin([0, 1, 2, 3])].copy()
validation = base[base.month.isin([5, 6])].copy()
test = base[base.month.eq(7)].copy()
pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(validation), len(test)],
    "fraud_rate": [train.fraud_bool.mean(), validation.fraud_bool.mean(), test.fraud_bool.mean()],
})

Month 4 is not reassigned. The test month is not used for preprocessing, model selection or threshold selection.

## 3. Candidate feature matrix

In [ ]:
excluded = [target, time_column, *audit_only]
model_columns = [column for column in base.columns if column not in excluded]
pd.DataFrame({"feature": model_columns, "dtype": base[model_columns].dtypes.astype(str).values})

## 4. Preprocessing boundaries

In [ ]:
from fraud_detection.modelling import build_logistic_pipeline, build_histogram_pipeline

logistic_pipeline = build_logistic_pipeline(train)
tree_pipeline = build_histogram_pipeline(train)
logistic_pipeline

Both pipelines fit imputers and encoders on training data only. The logistic model standardises numeric fields and one-hot encodes categories. The tree model uses ordinal codes for categories and leaves numeric scales unchanged.

## 5. Leakage register

In [ ]:
pd.DataFrame([
    ["model_score", "Excluded", "Author-provided prediction would leak an existing model into a new model"],
    ["fraud_bool", "Target only", "Outcome label"],
    ["month", "Split only", "Prevents random mixing across time"],
    ["audit-only fields", "Evaluation only", "Reserved for subgroup analysis"],
], columns=["field", "policy", "reason"])

## 6. Reproducibility checks

In [ ]:
assert set(train.month.unique()) == {0, 1, 2, 3}
assert set(validation.month.unique()) == {5, 6}
assert set(test.month.unique()) == {7}
assert "model_score" not in model_columns
assert not set(audit_only).intersection(model_columns)
print("Feature and split checks passed.")

## Conclusion

The feature policy is intentionally conservative. It separates model inputs, temporal controls and audit fields before any model is fitted.